# Milestone 1 Lab, notebook 3 of 6: Ask the LLM with all the data

## Where we are

| Notebook | What you do |
|---|---|
| 1. The business question | Read the brief, explore the data, and answer the question by hand |
| 2. Your first LLM calls | Call a model, get a structured reply, and have AI draft the metric definitions |
| **3. Ask the LLM with all the data** (you are here) | Put the data in the prompt, ask the question, and check the answer |
| 4. What is a tool, what is an agent | Build one tool, run the agent loop by hand, then let LangChain run it |
| 5. The agent answers the question | Give an agent six tools and compare it with the single call |
| 6. Verify and go to production | Check an answer with code, and see the production version |

Model requests in this notebook: **1**. With the default model, the whole lab costs a few US cents.

## What this notebook is for

Time to ask the leader's question. We take the simplest possible approach: paste the whole profit table into the prompt and ask.

Then we do what notebook 1 prepared us for: hold the model's answer against the answer key, line by line.

In [1]:
import json

import pandas as pd
from lab_helpers import (
    LAST_MONTH,
    MAIN_QUESTION,
    PREVIOUS_MONTH,
    AnalystAnswer,
    check_figures,
    compare_with_key,
    connect,
    definitions,
    flags,
    load_result,
    products,
    profit,
    save_result,
    show_answer,
    show_doc,
    structured_call,
)

client, MODEL = connect()

Connected to OpenRouter. Using model openrouter/free


`connect()` is the connection code you wrote in notebook 2, now in `lab_helpers.py`.

## Step 1: Recall the correct answer

This is the answer key you built by hand in notebook 1. A correct answer must contain these facts.

In [2]:
answer_key = load_result("answer_key", required_from="notebook 1")

print("Confidence should be:", answer_key["confidence"])
print("Data problems to mention:", answer_key["flags"])
pd.DataFrame(answer_key["figures"])[["what", "entity_id", "kind", "value"]]

Confidence should be: low
Data problems to mention: ['QF-001', 'QF-002']


,what,entity_id,kind,value
0,#1 product: Trail water bottle,P001,value,5188.20
1,#2 product: Headlamp,P008,value,4250.00
2,#3 product: Rain jacket,P009,value,3841.00
3,Top category: Hydration,Hydration,value,13358.06
4,"Hydration, change since the month before",Hydration,change,376.06
5,"Apparel, change since the month before (the bi...",Apparel,change,-5914.87
6,"Camp, change since the month before (cannot be...",Camp,change,NaN
7,Camp stove (its cost is missing),P007,value,NaN


## Step 2: Decide the shape of the answer

In notebook 2 we gave the model a small form to fill in. We do the same here with a bigger form, called `AnalystAnswer`.

The BRD says what a good answer contains. The form turns that list into fields.

In [3]:
show_doc("BRD.md", "5. What a good answer contains")

> **From `docs/BRD.md`: 5. What a good answer contains**

1. A headline that answers the question in one or two sentences.
2. The figures, each with the month, the product or category, and where it came from.
3. The metric definition used, with its version.
4. The assumptions that shape the figures.
5. Any known data problem that affects those figures.
6. A confidence level (high, medium, or low) with the reasons for it.

| Field in the form | What it holds |
|---|---|
| `headline` | The answer in a sentence or two |
| `narrative` | A short explanation |
| `figures` | Every number, each labelled with its metric, month, and product or category |
| `definitions_used` | Which metric definitions the answer relies on |
| `caveats` | Data problems that affect the figures |
| `confidence` | High, medium, or low, with reasons |
| `sources` | The tables the figures came from |
| `status`, `decline_reason` | Whether the question was answered or declined, and why |

The important one is `figures`. Because every number sits in its own labelled slot, code can check each one. The form is defined in `lab_helpers.py`, and the full detail is in section 5 of `docs/LLD.md`.

In [4]:
list(AnalystAnswer.model_fields)

['status',
 'headline',
 'narrative',
 'figures',
 'definitions_used',
 'assumptions',
 'caveats',
 'confidence',
 'sources',
 'decline_reason']

## Step 3: Build the prompt

The system prompt has two parts. First, the instructions.

In [5]:
INSTRUCTIONS = f"""You are the profit analyst for Kestrel Ridge Outfitters, a fictional online retailer of outdoor gear.
Answer the retail leader's question about profit.

- "Last month" is {LAST_MONTH}. The month before is {PREVIOUS_MONTH}. These are the only months with data.
- Put every number you use in figures, with its metric, month, and entity_id (a product_id such as P001, or a category name).
- For a change between the two months, use kind "change" and month {LAST_MONTH}.
- List every quality flag that affects your figures in caveats, with its flag_id.
- Set confidence to low if a high severity flag affects your figures, medium for a medium severity flag, otherwise high.
- A blank value is unavailable. Report it as null. Never treat it as zero or estimate it.
- Fill in every part of the answer. The headline, the narrative, and the figures must not be empty.
"""

Second, the data. The model can't open our files, so we paste the tables into the prompt as text. This is literally what the model will read:

In [6]:
profit_as_text = profit.merge(products, on="product_id").to_csv(index=False)

print(profit_as_text[:600], "...")

month,product_id,units_sold,gross_sales,discounts,refunds,net_sales,product_cost,gross_profit,fulfillment_cost,shipping_cost,payment_fees,return_cost,contribution_profit,metric_version,product_name,category
2026-09,P001,420,16800.0,840.0,76.0,15884.0,7560.0,8324.0,1050.0,1680.0,478.8,12.0,5103.2,1.0.0,Trail water bottle,Hydration
2026-09,P002,260,9100.0,455.0,66.5,8578.5,3900.0,4678.5,650.0,1040.0,259.35,12.0,2717.15,1.0.0,Insulated flask,Hydration
2026-09,P003,95,8550.0,855.0,162.0,7533.0,4560.0,2973.0,285.0,570.0,230.85,16.0,1871.15,1.0.0,Hydration pack,Hydration
2026-09,P004,150,8250.0,0.0, ...


In [7]:
SYSTEM_PROMPT = f"""{INSTRUCTIONS}
# Metric definitions

{json.dumps(definitions["metrics"], indent=2)}

# Table: monthly_product_profit (one row per product per month)

{profit_as_text}

# Table: quality_flags

{flags.to_csv(index=False)}
"""

print(f"The system prompt is {len(SYSTEM_PROMPT.split()):,} words long")

The system prompt is 846 words long


Stop and think about what we are asking the model to do with that wall of text:

- to find the **top products**, it must sort 12 numbers
- to find the **top category**, it must add up four numbers, three times
- to say **what changed**, it must do that for both months and subtract
- and it must notice the **Camp trap** from notebook 1

A language model is built to predict text. It is not a calculator.

## Step 4: Ask

This is the same structured call as in notebook 2 (`client.chat.completions.parse` with a form), using the bigger form.

There is one practical difference. A model can fail at a form this size, and free models often do: they send it back with nothing filled in. So we make the call through `structured_call`, a small wrapper in `lab_helpers.py`. It makes the same `parse` call, checks that the form came back filled in, and tries again (up to three times) if it didn't. Each attempt is one model request.

In [8]:
print(MAIN_QUESTION)

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": MAIN_QUESTION}]

llm_answer = structured_call(client, MODEL, messages, AnalystAnswer)

print(f"The answer has {len(llm_answer.figures)} figures")

Which products and categories generated the most contribution profit last month, what changed, and how confident should I be in those numbers?
Attempt 1: the request failed (LengthFinishReasonError: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=8192, prompt_tokens=3557, total_tokens=11749, com)
Attempt 2: answered by baidu/ernie-5.5-flash-preview:free
The answer has 13 figures


> **If that cell ended in an error**, all three attempts failed. Run it again. If it keeps failing, check `MODEL_NAME` in your `.env`: the lab is tested with `anthropic/claude-haiku-5.5`.

Here is the answer, displayed the way the leader would read it.

In [9]:
show_answer(llm_answer)

### October contribution profit was led by Trail water bottle and Hydration; confidence is low because Camp and P007 profit are incomplete.
Last month, the top products by contribution profit were Trail water bottle (P001) at $5,188.20, up $85.00; Headlamp (P008) at $4,250.00, up $538.00; and Rain jacket (P009) at $3,841.00, down $2,234.30. At category level, Hydration generated $13,358.06, up $376.06, and Apparel generated $9,746.38, down $5,914.87. Camp cannot be fully measured for October because Camp stove (P007) contribution profit is null; September Camp contribution profit was $11,961.49. Confidence is low because QF-001 is high severity and affects Camp and P007, while QF-002 affects P001 and QF-003 affects the September P011 figure embedded in the Apparel change.

**Declined:**  Camp cannot be fully ranked because contribution profit is unavailable due missing product cost.
| Figure | Value | Month |
|---|---|---|
| October contribution profit | $5,188.20 | 2026-10 |
| Change in contribution profit | +$85.00 | 2026-10 |
| October contribution profit | $4,250.00 | 2026-10 |
| Change in contribution profit | +$538.00 | 2026-10 |
| October contribution profit | $3,841.00 | 2026-10 |
| Change in contribution profit | -$2,234.30 | 2026-10 |
| October contribution profit | $13,358.06 | 2026-10 |
| Change in contribution profit | +$376.06 | 2026-10 |
| October contribution profit | $9,746.38 | 2026-10 |
| Change in contribution profit | -$5,914.87 | 2026-10 |
| October contribution profit | unavailable | 2026-10 |
| October contribution profit | unavailable | 2026-10 |
| September contribution profit | $11,961.49 | 2026-09 |

- **Caveat QF-001:** Missing product cost for Camp stove in 2026-10 makes P007 contribution profit null and prevents a complete Camp category total.
- **Caveat QF-002:** Duplicate October orders for Trail water bottle may overstate P001 sales and profit.
- **Caveat QF-003:** The September Sun hat refund issue may affect P011 September contribution profit and therefore the Apparel change.

**Confidence: low.** QF-001 is high severity and affects Camp and P007 contribution profit. QF-002 is medium severity and affects P001. QF-003 is low severity and affects the September P011 figure used in the Apparel change.

It reads well. It is fluent and confident. **But is it right?** You cannot tell by reading it. A correct answer and a wrong answer look exactly the same.

## Step 5: Check it against the answer key

`compare_with_key` puts the correct answer beside what the model said, one line per fact.

- **yes**: the model gave this fact and got it right
- **NO**: the model gave it and got it wrong
- **not given**: the model left it out (or labelled it in a way we couldn't match). For a fact whose correct answer is "unavailable", leaving it out counts as right

In [10]:
compare_with_key(answer_key, {"LLM": llm_answer})

,What,Correct answer,LLM said,LLM right?
0,#1 product: Trail water bottle,"$5,188.20","$5,188.20",yes
1,#2 product: Headlamp,"$4,250.00","$4,250.00",yes
2,#3 product: Rain jacket,"$3,841.00","$3,841.00",yes
3,Top category: Hydration,"$13,358.06","$13,358.06",yes
4,"Hydration, change since the month before",+$376.06,+$376.06,yes
5,"Apparel, change since the month before (the bi...","-$5,914.87","-$5,914.87",yes
6,"Camp, change since the month before (cannot be...",unavailable,not given,no
7,Camp stove (its cost is missing),unavailable,unavailable,yes
8,Confidence,low,low,yes
9,Mentions data problem QF-001,yes,yes,yes


Look especially at three lines:

- the **category total** (did it add four numbers correctly?)
- the **Camp change** (did it fall into the trap and report a fall, or did it correctly leave it out or mark it unavailable?)
- the **confidence** (did it apply the rule?)

The table above only covers the facts in our key. The model may have given other numbers too. `check_figures` checks every number it gave against the truth.

In [11]:
check_figures(llm_answer)

,Figure,AI said,True figure,Right?
0,October contribution profit,"$5,188.20","$5,188.20",yes
1,Change in contribution profit,+$85.00,+$85.00,yes
2,October contribution profit,"$4,250.00","$4,250.00",yes
3,Change in contribution profit,+$538.00,+$538.00,yes
4,October contribution profit,"$3,841.00","$3,841.00",yes
5,Change in contribution profit,"-$2,234.30","-$2,234.30",yes
6,October contribution profit,"$13,358.06","$13,358.06",yes
7,Change in contribution profit,+$376.06,+$376.06,yes
8,October contribution profit,"$9,746.38","$9,746.38",yes
9,Change in contribution profit,"-$5,914.87","-$5,914.87",yes


## What we learned

Be honest about what you just saw. With a capable model such as the lab's default, every line is probably right: the sums, the changes, even the Camp trap. A good model can add up four numbers.

So why not stop here and ship this? Because being right this time is not the same as being trustworthy. This approach has four problems that don't go away:

1. **You can't tell if it's right without an answer key.** We only know because we did the work by hand first. For a new question, there is no key.
2. **You can't show where a number came from.** The model read a wall of text and produced a figure. There is nothing to point to if finance asks "where is this from?"
3. **It doesn't scale.** We pasted 24 rows into the prompt, and the model has to read all of them for every question. A real retailer has millions of rows. They will not fit, and the more you paste, the slower, costlier, and less accurate it gets.
4. **Nothing guarantees it.** The rules (never treat a missing cost as zero, don't compare Camp) are requests in a prompt. A weaker model, a longer table, or an unlucky run can ignore them, and the answer will look just as confident. Try `MODEL_NAME=openrouter/free` some time and compare.

A profit figure that is confidently wrong is worse than no figure. Imagine the leader cutting Camp's budget because of a "fall" that was really one product's missing cost.

So here is the question for the next notebook: can we keep what the model is good at (understanding the question, explaining the result) and take the arithmetic away from it?

Save this answer so we can compare it with the agent's later.

In [12]:
save_result("llm_answer", llm_answer.model_dump())

Saved results/llm_answer.json


**Next: notebook 4**, where we meet tools and agents.